In [1]:
import glob
import os
import time

import cv2
import numpy as np
import torch
from cellpose import models, transforms
from scipy import ndimage

TESTDATA_DIR = r"F:\Livo\Data - 2026\1753802296\testdata"
OUT_DIR = r"F:\Livo\Data - 2026\1753802296\cellpose_rbc_contours_out_new"
N_IMAGES = 30
DOWNSAMPLE = 2 
DIAMETER = 30.0 
NITER = 200 
BATCH_SIZE = 16  
USE_FP16 = True  

os.makedirs(OUT_DIR, exist_ok=True)

## Load the model

Pretrained generalist weights -- no training step. `gpu=True` only actually uses the GPU
if a CUDA-enabled torch build is installed; otherwise Cellpose silently falls back to CPU.

In [2]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("!! No CUDA torch found -- install it first (see prereq above). "
          "Falling back to CPU; timings will NOT reflect the GPU path.")

model = models.Cellpose(gpu=torch.cuda.is_available(), model_type="cyto3")
cp = model.cp 

CUDA available: True
Device: NVIDIA GeForce RTX 3050


## Benchmark loop -- with network-vs-dynamics timing split



In [3]:
import json

paths = sorted(glob.glob(os.path.join(TESTDATA_DIR, "*.jpg")))[:N_IMAGES]
print(f"Benchmarking on {len(paths)} FOVs from {TESTDATA_DIR} at {DOWNSAMPLE}x downsample, "
      f"niter={NITER}, batch_size={BATCH_SIZE}, fp16={USE_FP16}")

rescale = cp.diam_mean / DIAMETER
net_times, dyn_times, contour_times, total_times = [], [], [], []

for i, path in enumerate(paths):
    name = os.path.splitext(os.path.basename(path))[0]
    img_bgr = cv2.imread(path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    infer_rgb = cv2.resize(img_rgb, (w // DOWNSAMPLE, h // DOWNSAMPLE), interpolation=cv2.INTER_AREA)

    x = transforms.convert_image(infer_rgb, [0, 0], channel_axis=None, z_axis=None,
                                  do_3D=False, nchan=cp.nchan)
    if x.ndim < 4:
        x = x[np.newaxis, ...]
    x = transforms.normalize_img(x)

    t0 = time.time()
    if USE_FP16:
        with torch.autocast("cuda", dtype=torch.float16):
            dP, cellprob, styles = cp._run_net(x, rescale=rescale, batch_size=BATCH_SIZE)
    else:
        dP, cellprob, styles = cp._run_net(x, rescale=rescale, batch_size=BATCH_SIZE)
    t_net = time.time() - t0

    t0 = time.time()
    masks = cp._compute_masks(x.shape, dP, cellprob, niter=NITER)
    t_dyn = time.time() - t0

    masks_full = cv2.resize(masks.astype(np.int32), (w, h), interpolation=cv2.INTER_NEAREST)
    n_instances = int(masks_full.max())

   
    t0 = time.time()
    slices = ndimage.find_objects(masks_full)
    overlay = img_bgr.copy()
    instances_json = []
    for inst_id, sl in enumerate(slices, start=1):
        if sl is None:
            continue
        ys, xs = sl
        y0, y1 = max(ys.start - 1, 0), min(ys.stop + 1, h)
        x0, x1 = max(xs.start - 1, 0), min(xs.stop + 1, w)
        crop = (masks_full[y0:y1, x0:x1] == inst_id).astype(np.uint8)
        contours, _ = cv2.findContours(crop, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in contours:
            c = c + np.array([x0, y0])  # shift back to full-image coordinates
            cv2.drawContours(overlay, [c], -1, (0, 255, 0), 1)
            instances_json.append({"id": inst_id, "contour": c.reshape(-1, 2).astype(int).tolist()})
    t_contour = time.time() - t0

    dt = t_net + t_dyn + t_contour

    cv2.imwrite(os.path.join(OUT_DIR, f"{name}_contours.png"), overlay)
    with open(os.path.join(OUT_DIR, f"{name}_contours.json"), "w") as f:
        json.dump({
            "image": name,
            "image_size": {"width": w, "height": h},
            "num_instances": n_instances,
            "instances": instances_json,
        }, f)

    tag = "(includes CUDA warmup)" if i == 0 else ""
    print(f"  {name}: net={t_net*1000:.1f}ms  dynamics={t_dyn*1000:.1f}ms  "
          f"contours={t_contour*1000:.1f}ms  total={dt*1000:.1f}ms  {n_instances} instances {tag}")
    if i > 0:  # skip first call's one-time CUDA/autocast kernel-selection warmup
        net_times.append(t_net)
        dyn_times.append(t_dyn)
        contour_times.append(t_contour)
        total_times.append(dt)

if total_times:
    n = len(total_times)
    avg_net = 1000 * sum(net_times) / n
    avg_dyn = 1000 * sum(dyn_times) / n
    avg_contour = 1000 * sum(contour_times) / n
    avg_total = 1000 * sum(total_times) / n
    print(f"\nAverage (excluding warmup call) over {n} images:")
    print(f"  network forward:  {avg_net:.1f} ms/image ({100*avg_net/avg_total:.0f}%)")
    print(f"  dynamics:         {avg_dyn:.1f} ms/image ({100*avg_dyn/avg_total:.0f}%)")
    print(f"  contour extract:  {avg_contour:.1f} ms/image ({100*avg_contour/avg_total:.0f}%)")
    print(f"  total:            {avg_total:.1f} ms/image")
print(f"Contour overlays + JSON saved to: {OUT_DIR}")

Benchmarking on 30 FOVs from F:\Livo\Data - 2026\1753802296\testdata at 2x downsample, niter=200, batch_size=16, fp16=True
  Img_0_13: net=492.8ms  dynamics=1610.7ms  contours=20.0ms  total=2123.4ms  225 instances (includes CUDA warmup)
  Img_0_15: net=163.5ms  dynamics=331.6ms  contours=15.0ms  total=510.1ms  216 instances 
  Img_0_17: net=169.6ms  dynamics=352.1ms  contours=125.7ms  total=647.4ms  213 instances 
  Img_0_2: net=162.6ms  dynamics=325.1ms  contours=17.0ms  total=504.7ms  237 instances 
  Img_0_21: net=161.7ms  dynamics=317.6ms  contours=18.0ms  total=497.3ms  198 instances 
  Img_0_23: net=164.2ms  dynamics=362.8ms  contours=16.0ms  total=543.0ms  194 instances 
  Img_0_32: net=162.4ms  dynamics=289.8ms  contours=15.0ms  total=467.2ms  174 instances 
  Img_0_35: net=162.0ms  dynamics=414.5ms  contours=14.6ms  total=591.1ms  170 instances 
  Img_0_36: net=163.9ms  dynamics=293.3ms  contours=15.0ms  total=472.2ms  171 instances 
  Img_0_37: net=166.4ms  dynamics=298.0ms  